# Eve SFT — continue an adapter (v12 → v13)

v12 trained a full epoch with a masking bug: the closing `<turn|>` marker — Gemma 4's EOS — was excluded from the loss, so the adapter never learned to stop and ran every eval reply to the token cap. The fix is in `train/masking.py`. Rather than spend another ~9 h of GPU quota on a fresh epoch, this notebook continues v12's adapter for a short run with the fixed labels, then checks that replies end on their own before anything is downloaded.

**Inputs:** `therapylink-corpus` (data + code) and `therapylink-eve-sft` (the v12 adapter). **Accelerator:** one T4, internet on.

In [ ]:
# Same environment as eve_sft_kaggle.ipynb: transformers>=5.5.2 for Gemma 4's KV sharing under
# gradient checkpointing (Kaggle's image ships 5.5.0).
%pip install -q unsloth trl bitsandbytes typer rich pydantic
%pip install -q -U "transformers>=5.5.2"

import glob, os, shutil

WORK = '/kaggle/working/tl'
hits = glob.glob('/kaggle/input/**/code/therapylink_ml', recursive=True)
assert hits, 'No therapylink_ml found. Attach the therapylink-corpus dataset.'
shutil.copytree(os.path.dirname(hits[0]), WORK + '/ml', dirs_exist_ok=True)
os.chdir(WORK + '/ml')
%pip install -q -e . --no-deps

train = glob.glob('/kaggle/input/**/conversations.train.jsonl', recursive=True)[0]
os.makedirs('data/processed', exist_ok=True)
shutil.copy(train, 'data/processed/conversations.train.jsonl')

# Kaggle flattens a dataset's single top-level folder, so the adapter files may sit at the dataset
# root rather than under adapter/. Find any directory holding a complete PEFT adapter.
adapters = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(f), 'adapter_model.safetensors'))]
assert adapters, 'No LoRA adapter found. Attach the therapylink-eve-sft dataset.'
ADAPTER = adapters[0]
print('adapter:', ADAPTER)
!python -c "import transformers, trl; print('transformers', transformers.__version__, '| trl', trl.__version__)"

In [ ]:
# 250 steps at ~19.4 s/step on a T4 in fp32 is ~1.4 h. The adapter already carries a full epoch
# of content; what it lacks is where to stop, and that single-token pattern is learned quickly.
# Lower LR so the content isn't disturbed; a new seed so these steps see a different slice of the
# shuffled corpus than v12's opening steps did.
%env CUDA_VISIBLE_DEVICES=0
%env HF_DATASETS_DISABLE_CACHING=1
!python -u -m therapylink_ml.train.eve_sft \
  --model-name unsloth/gemma-4-E2B-it \
  --init-adapter {ADAPTER} \
  --weights esconv=2,annomi=3,counselchat=1.5,empathetic_dialogues=0.15 \
  --max-steps 250 --lr 5e-5 --warmup 0.04 --seed 14 \
  --max-hours 3 --no-gguf --out /kaggle/working/runs/eve-sft-e2b-v13

In [ ]:
# Fails the version if any reply runs on to the cap, so a bad adapter is visible from the log alone.
!python -u -m therapylink_ml.train.eos_check /kaggle/working/runs/eve-sft-e2b-v13/adapter

In [ ]:
!ls -la /kaggle/working/runs/eve-sft-e2b-v13/adapter 2>/dev/null || ls -la /kaggle/working/runs/